# 09c The personal map: ablations and interpretation (THINGS-EEG2)

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 01 and 03b, and the THINGS-EEG2 dataset `cross-subject-brain-encoding-eeg` (09g).

Development analysis, not a prespecified test. The design of 09b on THINGS-EEG2: the map against borrowing, Gu's ensemble, the converter and own ridge at
k = 50 to 2,000; the map's window and rank; the size of the correction per window; and whether a person's map is
more similar to itself than to other people's maps.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '09c-h4bablate', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"k_curve": [50, 100, 250, 500, 1000, 2000], "draws": [13, 14, 15], "window_s": [0.1, 0.2], "map_windows_ms": {"50": 50, "100": 100, "200": 200, "epoch": null}, "ranks": [1, 2, 4, 8], "ablation_ks": [250, 500], "interpret_k": 500, "half": 1000, "dataset": "EEG2"}')
TAG, DATASET = '09c', 'EEG2'
PEOPLE = ['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10']

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.eval import eeg1_h4b as h4b
from cross_subject_encoding.eval import h4b_analysis as ha
from cross_subject_encoding.eval.paper_correlation import paper_correlation
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.models.baselines import best_source, gu_by_group, window_groups
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.kaggle_export import export_to_dataset

import shutil
from cross_subject_encoding.data.splits import subset
with np.load(C.find_artifact("paper_alexnet_kpca1000.npz")) as f:
    z_train, z_test = f["train"].astype(np.float64), f["test"].astype(np.float64)
subsets = json.loads(C.find_artifact(C.EEG_K_SUBSETS).read_text())
people = PEOPLE
ATTACHED = te.attached(C.EEG_RAW)  # the 09g dataset (flat names) or the release layout
WORK = C.EEG_RAW if ATTACHED else Path("/kaggle/tmp/things_eeg2")
y_train, bio_test = {}, {}
for s in people:
    if not ATTACHED and te.subject_file(WORK, s, "test") is None:
        te.download_subject(s, WORK)
    tr = te.load_part(te.subject_file(WORK, s, "training"))
    y_train[s], times = tr["data"].mean(1).astype(np.float32), tr["times"]
    bio_test[s] = te.load_part(te.subject_file(WORK, s, "test"))["data"].astype(np.float32)
    del tr
    if not ATTACHED:
        shutil.rmtree(WORK / s, ignore_errors=True)
order = lambda s, d, k: np.asarray(subset(subsets, s, d, k))
print(len(people), 'people loaded |', f'{(time.monotonic() - T0) / 60:.1f} min')

In [ ]:
ALPHAS = [10.0 ** e for e in range(-2, 9)]
N_TRAIN, N_TEST = len(z_train), len(z_test)
n_ch, n_t = 17, 100
query = np.concatenate([z_train, z_test])
src_train = np.empty((len(people), N_TRAIN, n_ch * n_t), np.float32)
src_test = np.empty((len(people), N_TEST, n_ch * n_t), np.float32)
for i, s in enumerate(people):
    pred = pl.ols_paper(z_train, y_train[s].reshape(N_TRAIN, -1), query)
    src_train[i], src_test[i] = pred[:N_TRAIN], pred[N_TRAIN:]
groups = window_groups(times, n_ch, C.EEG_WINDOW_MS)
window = (times >= FIXED["window_s"][0]) & (times < FIXED["window_s"][1])
win = {name: ha.window_index(times, ms) for name, ms in FIXED["map_windows_ms"].items()}
rows, norms, half_a, half_b = [], [], {}, {}
for i, person in enumerate(people):
    t = time.monotonic()
    others = [j for j in range(len(people)) if j != i]
    avg_train, avg_test = src_train[others].mean(0), src_test[others].mean(0)
    avg_t3 = avg_test.reshape(N_TEST, n_ch, n_t)
    preds = {"avg|all|all|-": avg_t3}
    for d in FIXED["draws"]:
        for k in FIXED["k_curve"]:
            idx = order(person, d, k)
            target = y_train[person][idx]
            flat = target.reshape(k, -1)
            a_k = avg_train[idx].reshape(k, n_ch, n_t)
            coefs, _ = h4b.fit_map(a_k, target, win["100"], seed=d)
            preds[f"h4b|{k}|{d}|w100r17"] = h4b.apply_map(coefs, avg_t3, win["100"])
            preds[f"own_ridge|{k}|{d}|-"] = RidgePerOutput(ALPHAS, C.RIDGE_FOLDS, d).fit(z_train[idx], flat).predict(z_test)
            borrowed = src_train[np.ix_(others, idx)]
            best = best_source(list(borrowed), flat)
            preds[f"converter|{k}|{d}|-"] = RidgePerOutput(ALPHAS, C.RIDGE_FOLDS, d).fit(borrowed[best], flat).predict(src_test[others][best])
            preds[f"gu_window|{k}|{d}|-"] = gu_by_group(borrowed, flat, src_test[others], groups, "pooled")
            if k in FIXED["ablation_ks"]:
                for name in ("50", "200", "epoch"):
                    c_w, _ = h4b.fit_map(a_k, target, win[name], seed=d)
                    preds[f"h4b|{k}|{d}|w{name}r17"] = h4b.apply_map(c_w, avg_t3, win[name])
                for r in FIXED["ranks"]:
                    preds[f"h4b|{k}|{d}|w100r{r}"] = h4b.apply_map(ha.truncate_rank(coefs, r), avg_t3, win["100"])
            if k == FIXED["interpret_k"] and d == FIXED["draws"][0]:
                norms.append({"person": person, **{f"w{w}": v for w, v in ha.map_norms(coefs).items()}})
    o = order(person, FIXED["draws"][0], 2 * FIXED["half"])
    for store, part in ((half_a, o[:FIXED["half"]]), (half_b, o[FIXED["half"]:])):
        store[person] = h4b.fit_map(avg_train[part].reshape(len(part), n_ch, n_t), y_train[person][part], win["100"], seed=13)[0]
    scored = paper_correlation({n: np.asarray(p).reshape(N_TEST, n_ch, n_t) for n, p in preds.items()}, bio_test[person], n_iter=100)
    for name, curve in scored["correlation"].items():
        method, k, d, variant = name.split("|")
        rows.append({"person": person, "method": method, "k": k, "draw": d, "variant": variant,
                     "r_100_200ms": float(curve[:, window].mean())})
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / f"{TAG}_scores.csv", index=False)
    print(person, f"{time.monotonic() - t:.0f} s")

In [ ]:
frame = pd.DataFrame(rows)
m = frame.groupby(["person", "method", "k", "variant"])["r_100_200ms"].mean()
avg = {p: m[(p, "avg", "all", "-")] for p in people}
def mean_over(method, k, variant):
    return float(np.mean([m[(p, method, str(k), variant)] for p in people]))
summary = {"people": len(people), "k_curve": {}, "windows": {}, "ranks": {}}
for k in FIXED["k_curve"]:
    summary["k_curve"][k] = {"avg": float(np.mean(list(avg.values()))), "h4b": mean_over("h4b", k, "w100r17"),
                             "gu_window": mean_over("gu_window", k, "-"), "converter": mean_over("converter", k, "-"),
                             "own_ridge": mean_over("own_ridge", k, "-")}
for k in FIXED["ablation_ks"]:
    summary["windows"][k] = {name: mean_over("h4b", k, f"w{name}r17") for name in FIXED["map_windows_ms"]}
    summary["ranks"][k] = {str(r): mean_over("h4b", k, f"w100r{r}") for r in FIXED["ranks"] + [17]}
nf = pd.DataFrame(norms)
summary["map_norm_by_window_ms"] = {f"{-200 + 100 * int(c[1:])}": float(nf[c].mean()) for c in nf.columns if c.startswith("w")}
rep = ha.reproducibility(half_a, half_b)
between_by_person = {p: float(np.nanmean([np.nanmean(list(ha.cosine(half_a[p], half_a[q]).values())) for q in people if q != p])) for p in people}
diff = np.array([rep["within_by_person"][p] - between_by_person[p] for p in people])
summary["reproducibility"] = {"within_by_window": rep["within"], "between_by_window": rep["between"],
                              "within_minus_between_mean": float(diff.mean()), "positive": int((diff > 0).sum()),
                              "p_one_sided": float(stats.ttest_1samp(diff, 0, alternative="greater").pvalue)}
(C.METRICS_DIR / f"{TAG}_summary.json").write_text(json.dumps(summary, indent=2))
np.savez_compressed(C.METRICS_DIR / f"{TAG}_maps_k1000.npz", people=np.array(people),
                    half_a=np.stack([np.stack([half_a[p][w] for w in sorted(half_a[p])]) for p in people]),
                    half_b=np.stack([np.stack([half_b[p][w] for w in sorted(half_b[p])]) for p in people]))
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for meth in ("avg", "h4b", "gu_window", "converter", "own_ridge"):
    ax[0].plot(FIXED["k_curve"], [summary["k_curve"][k][meth] for k in FIXED["k_curve"]], marker="o", label=meth)
ax[0].set(xscale="log", xlabel="labelled images k", ylabel="r, 100-200 ms", title=f"{DATASET}: methods by k")
ax[0].legend(fontsize=7)
ms = sorted(summary["map_norm_by_window_ms"], key=int)
ax[1].bar([int(x) + 50 for x in ms], [summary["map_norm_by_window_ms"][x] for x in ms], width=80)
ax[1].set(xlabel="window centre (ms)", ylabel="correction size (Frobenius)", title="When the personal map matters")
w_ms = [-200 + 100 * w + 50 for w in sorted(rep["within"])]
ax[2].plot(w_ms, [rep["within"][w] for w in sorted(rep["within"])], marker="o", label="same person, two halves")
ax[2].plot(w_ms, [rep["between"][w] for w in sorted(rep["between"])], marker="o", label="different people")
ax[2].set(xlabel="window centre (ms)", ylabel="map cosine similarity", title="Are the maps personal?")
ax[2].legend(fontsize=7)
fig.tight_layout()
fig.savefig(C.METRICS_DIR / f"{TAG}_figure.png", dpi=150)
print(json.dumps({k: v for k, v in summary.items() if k != "reproducibility"}, indent=1)[:2500])
print(json.dumps(summary["reproducibility"], indent=1)[:1500])

In [ ]:
export_to_dataset(slug="cross-subject-brain-encoding-09c-outputs", title="XSUB 09c H4b ablation outputs",
                  files=[C.METRICS_DIR / f"{TAG}_{x}" for x in ("summary.json", "scores.csv", "maps_k1000.npz", "figure.png")],
                  include_dirs=[], message="h4b ablation and interpretation")